# 03 — Mass and Balance

Shared `MassBudget` with component positions. Everything except the
battery is fixed by the airframe YAML; the battery is slid fore/aft to put
the CG on the target from notebook 02, then checked against the pod.

Payloads come from the **shared** `payload/payload_manifest.yaml`; the
multirotor's always-on compute stack is excluded on this airframe.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "flying_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "flying_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

from uavdesign.atmosphere import G, isa_density, wing_loading_g_dm2, wing_loading_n_m2
from uavdesign.aero import Wing, DragPolar, FixedWingAero

alt_m = mission["mission"]["altitude_asl_m"]
rho = isa_density(alt_m, mission["environment"]["temperature_offset_c"])
wing = Wing.from_dict(airframe["wing"])
aero = design["aero"]
perf = design["performance"]
mass_kg = design["design_point"]["target_auw_kg"]
static_margin = design["design_point"]["static_margin"]
polar = DragPolar.from_oswald(aero["cd0"], wing.aspect_ratio, aero["oswald_e"], aero["cl_max"])

print(f"Site: {alt_m} m ASL, rho = {rho:.4f} kg/m^3")
print(f"Wing: S = {wing.area_dm2:.2f} dm^2, b = {wing.span_mm:.0f} mm, AR = {wing.aspect_ratio:.2f}")
print(f"Polar: CD0 = {polar.cd0}, k = {polar.k:.4f}, CLmax = {polar.cl_max}")


In [ ]:
from uavdesign.mass import MassBudget

payloads = design["payloads"]
batt = design["battery"]
bay = airframe["battery_bay_x_mm"]

mb = MassBudget.from_yaml(
    PLATFORM_DIR / "airframe_params.yaml",
    REPO_ROOT / "payload" / "payload_manifest.yaml",
    active_payloads=payloads,
    include_always_on=False,
    payload_positions_mm={n: design["payload_x_mm"] for n in payloads},
)
mb.add("battery", batt["mass_g"])

x_cg_target = wing.cg_for_static_margin(static_margin)
x_batt_ideal = mb.solve_position_for_cg("battery", x_cg_target)
x_batt = float(np.clip(x_batt_ideal, *bay))
mb.set_position("battery", x_batt)
sm_actual = wing.static_margin(mb.cg_x_mm())

print(f"Target CG:            {x_cg_target:6.1f} mm (SM {static_margin:.1%})")
print(f"Battery position:     {x_batt_ideal:6.1f} mm ideal, bay {bay[0]}-{bay[1]} mm")
print(f"Resulting CG:         {mb.cg_x_mm():6.1f} mm (SM {sm_actual:.1%})")
print(f"AUW:                  {mb.auw_g:6.0f} g  (target {mass_kg*1000:.0f} g, "
      f"margin {mass_kg*1000 - mb.auw_g:+.0f} g)")


## Ballast check

If the battery cannot reach the ideal position, this is the nose (or
tail) ballast needed at the bay end to restore the target margin.


In [ ]:
x_now, m_now = mb.cg_x_mm(), mb.dry_mass_g
if np.isclose(x_batt, x_batt_ideal):
    print("Battery reaches the ideal position — no ballast needed.")
else:
    x_ballast = bay[0] if x_batt_ideal < bay[0] else bay[1]
    ballast_g = m_now * (x_now - x_cg_target) / (x_cg_target - x_ballast)
    print(f"Ballast needed: {ballast_g:.0f} g at x = {x_ballast} mm")


## Component table


In [ ]:
import pandas as pd

rows = [
    {"component": n, "mass_g": m, "x_mm": mb.positions_mm[n],
     "moment_g_mm": m * mb.positions_mm[n]}
    for n, m in mb.components_g.items()
]
table = pd.DataFrame(rows).sort_values("mass_g", ascending=False)
table["fraction_of_auw_pct"] = (table["mass_g"] / mb.auw_g * 100).round(1)
table.round(1)


## Static margin vs battery position

The slope shows how sensitive trim is to where the pack sits — on a
sub-kilogram wing a few millimetres matter.


In [ ]:
xs = np.linspace(bay[0] - 30, bay[1] + 30, 200)
sm_curve = []
for xb in xs:
    mb.set_position("battery", xb)
    sm_curve.append(wing.static_margin(mb.cg_x_mm()))
mb.set_position("battery", x_batt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, np.array(sm_curve) * 100)
ax.axvspan(*bay, color="C1", alpha=0.15, label="battery bay")
ax.axhline(static_margin * 100, color="k", ls="--", lw=1, label="target SM")
ax.axhline(0, color="C3", lw=1, label="neutral (unstable aft)")
ax.plot(x_batt, sm_actual * 100, "o", color="k")
ax.set_xlabel("Battery CG position [mm aft of root LE]")
ax.set_ylabel("Static margin [% MAC]")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_sm_vs_battery_position.png", dpi=140)
plt.show()


## Mass tornado (shared model)


In [ ]:
rows = mb.tornado(fraction=0.10)
names = [r[0] for r in rows]
lows = np.array([r[1] for r in rows]) * 1000
highs = np.array([r[2] for r in rows]) * 1000

fig, ax = plt.subplots(figsize=(7, 0.35 * len(names) + 1.5))
yy = np.arange(len(names))
ax.barh(yy, highs - lows, left=lows, color="C0", alpha=0.8)
ax.axvline(mb.auw_g, color="k", ls="--", lw=1, label=f"baseline {mb.auw_g:.0f} g")
ax.set_yticks(yy)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel("AUW [g]")
ax.set_title("Mass tornado — +/- 10 % per component")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_mass_tornado.png", dpi=140)
plt.show()


## Outputs handed to notebooks 04/05


In [ ]:
balance_summary = {
    "auw_kg": mb.auw_kg,
    "battery_x_mm": x_batt,
    "cg_x_mm": mb.cg_x_mm(),
    "static_margin": sm_actual,
    "mass_without_battery_g": mb.dry_mass_g - batt["mass_g"],
    "contingency_fraction": mb.contingency_fraction,
}
balance_summary
